# 03 — H1-N controlled training and analysis

H1-N is a narrow representation comparison: FFT-magnitude ResNet-50 versus RGB ResNet-50, both trained from random initialisation, on the same source-normalised 128 × 128 raster and paired-group training stream. The original direct-resize experiments are D0 diagnostics and are deliberately excluded from this comparison.

The six commands below are long-running. They are printed by default and run only when `RUN_H1N_NEURAL_TRAINING=1` is set in the notebook kernel environment. This prevents accidental heavy training while keeping the cells locally runnable.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL, preprocessing_metadata
from ai_image_detector.manifest import load_manifest

def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'scripts' / 'run_experiment.py').is_file():
            return candidate
    raise RuntimeError('Open the notebook from the repository or one of its subdirectories.')

REPO = find_repository_root()
MANIFEST = REPO / 'data/processed/defactify_grouped/manifest.csv'
assert MANIFEST.is_file(), 'Run the grouped-split preparation before H1-N training.'
frame = load_manifest(MANIFEST, check_paths=True)
assert set(['train', 'val', 'test']).issubset(set(frame.split))
assert preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)['image_size'] == 128

print(f'Repository: {REPO}')
print(frame.groupby(['split', 'label']).size().rename('n'))

## D0 record: read-only diagnostic evidence

The old radial-FFT and file-metadata artifacts may be inspected to document why the protocol changed. They must not enter a H1-N ranking, threshold choice, model card, or web interface. The metadata control receives geometry/source information unavailable to the intended image model; its score is evidence of dataset bias, not detection quality.

In [ ]:
d0_rows = []
for run_name in ('radial_logistic_seed7', 'file_metadata_control_seed7'):
    metrics_path = REPO / 'artifacts' / run_name / 'internal_test_metrics.json'
    if metrics_path.is_file():
        d0_rows.append({'status': 'D0 diagnostic only', 'run': run_name, **json.loads(metrics_path.read_text())})

d0_table = pd.DataFrame(d0_rows)
if d0_table.empty:
    print('No D0 artifact was found. That is not an H1-N result.')
else:
    display(d0_table)

## Predeclared H1-N neural runs

All six runs have equal epoch cap, batch size, optimizer family, early-stopping rule and validation-only threshold selection. `--from-scratch` is explicit: ImageNet pretraining is excluded because it is an RGB semantic prior without an equivalent FFT interpretation. The CLI default preprocessing is intentionally left unset in the command: its default is the locked `h1n_square_crop_128_v1` protocol, which selects the 128 × 128 square-crop raster and paired group sampler.

In [ ]:
SEEDS = (7, 17, 42)
REPRESENTATIONS = ('rgb', 'fft')

def experiment_dir(representation: str, seed: int) -> Path:
    return REPO / 'artifacts' / f'h1n_{representation}_resnet50_seed{seed}'

def train_command(representation: str, seed: int) -> list[str]:
    return [
        sys.executable,
        'scripts/run_experiment.py',
        '--manifest', 'data/processed/defactify_grouped/manifest.csv',
        '--representation', representation,
        '--output-dir', str(experiment_dir(representation, seed).relative_to(REPO)),
        '--seed', str(seed),
        '--epochs', '15',
        '--batch-size', '32',
        '--learning-rate', '0.0001',
        '--patience', '4',
        '--from-scratch',
    ]

TRAIN_COMMANDS = [train_command(representation, seed) for representation in REPRESENTATIONS for seed in SEEDS]
for command in TRAIN_COMMANDS:
    print(shlex.join(command))

if os.environ.get('RUN_H1N_NEURAL_TRAINING') == '1':
    for command in TRAIN_COMMANDS:
        subprocess.run(command, check=True, cwd=REPO)
else:
    print('PENDING: set RUN_H1N_NEURAL_TRAINING=1 only to launch the predeclared long runs.')

## Cluster-aware evaluation and paired comparison

After all six saved prediction files exist, analyse each run with leakage-group cluster bootstrap intervals. For every seed, the FFT command compares FFT minus RGB. The analysis reports per-generator slices, paired-group ranking accuracy and confidence intervals without selecting hyperparameters on the test rows. The original test has already been inspected in D0, so these are still exploratory internal stress-test analyses.

In [ ]:
def analysis_command(representation: str, seed: int) -> list[str]:
    command = [
        sys.executable,
        'scripts/analyze_predictions.py',
        '--experiment-dir', str(experiment_dir(representation, seed).relative_to(REPO)),
        '--bootstrap-repeats', '2000',
        '--seed', '20260829',
    ]
    if representation == 'fft':
        command.extend(['--compare-to', str(experiment_dir('rgb', seed).relative_to(REPO))])
    return command

ANALYSIS_COMMANDS = [analysis_command(representation, seed) for representation in REPRESENTATIONS for seed in SEEDS]
for command in ANALYSIS_COMMANDS:
    print(shlex.join(command))

if os.environ.get('RUN_H1N_ANALYSIS') == '1':
    for command in ANALYSIS_COMMANDS:
        subprocess.run(command, check=True, cwd=REPO)
else:
    print('PENDING: run only after all matched H1-N experiment directories are complete.')

## H1-N decision rule

Report ROC-AUC, PR-AUC, balanced accuracy, macro-F1, class recalls, FPR at TPR 95%, per-generator values and paired group-ranking accuracy. Aggregate the three *predeclared* seeds; do not select a lucky seed. The only confirmatory evaluation is the locked Synthbuster + RAISE external corpus after the model family, preprocessing, seed aggregation, checkpoint rule and threshold rule have all been frozen.